<a href="https://colab.research.google.com/github/BlairRong/Python-AI-course-lab-Cloud/blob/main/4.13_lab_AzureCosmosDB_FitnessApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lab 1 – Fitness App (Design + Cosmos DB Implementation)

1. Data the system needs:

User profile: userId, name, age, weight, registration date, etc.

Workout sessions: each session has date, type (running, cycling, etc.), duration (minutes), calories burned.

User goals (optional): target weight, weekly workout goal, etc.



2. What should be stored together in one document?

User profile + all workouts should be stored together in the same document.

Reason: The most common access pattern is “show a user’s profile and their recent workout history”. Keeping them together allows a single read operation to fetch everything needed.



3. What should be separated?

Global leaderboards or aggregated statistics (e.g., total calories burned by all users) could be stored in a separate container or computed separately.

Reason: These are not per-user access patterns; they require cross-partition queries and would benefit from a separate design (e.g., a materialized view or a different container).

Large media files (e.g., user profile pictures, workout photos) – these should be stored in Azure Blob Storage, not in Cosmos DB.

User goals could be embedded in the same document if small, but if they change frequently and are read separately, they could be split into a nested object. For simplicity, they can stay embedded.




Part 1 – System Design (Fitness App)
1. What are your main access patterns?

Retrieve a user’s full profile (including all their workouts) by userId.

Get all users who performed a specific type of workout (e.g., "running").

Get the oldest users (order by age).

Count total number of users.



2. What data do you need to retrieve together?

For a user, all their workouts (as a nested array) should be retrieved together.

The user’s personal info (name, age, weight) is stored alongside their workouts to avoid separate queries.



In [ ]:
!pip install azure-cosmos

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.9/424.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 13.7 MB/s eta 0:00:00


In [ ]:
import os
from azure.cosmos import CosmosClient, PartitionKey
import random
import datetime

In [ ]:
# Read from environment variables
url = os.environ.get('COSMOS_DB_URL', '')
key = os.environ.get('COSMOS_DB_KEY', '')

client = CosmosClient(url, credential=key)

In [ ]:
#Part2 - create Cosmos DB structure
# create database 创建数据库（如果不存在）
database_name = "fitnessDB"
try:
    database = client.create_database_if_not_exists(id=database_name)
    print(f"Database '{database_name}' ready")
except Exception as e:
    print(e)

Database 'fitnessDB' ready


In [ ]:
# create container 创建容器（如果不存在）
container_name = "users"
partition_key_path = "/userId"
try:
    container = database.create_container_if_not_exists(
        id=container_name,
        partition_key=PartitionKey(path=partition_key_path),
        offer_throughput=400
    )
    print(f"Container '{container_name}' ready")
except Exception as e:
    print(e)

Container 'users' ready


1. Why did you choose this partition key (/userId)?

I chose /userId because most queries are per user (e.g., fetching a user’s profile and workouts). This ensures all documents for the same user are stored in the same partition, making reads efficient and cost-effective.


2. How will it affect scalability?

With /userId as partition key, the data is evenly distributed across partitions as long as the number of users is large. Each user’s data stays together, which is good for read performance. However, if a single user generates an extremely high number of workouts, that partition could become a “hot partition”. For a fitness app with typical usage, this is unlikely, and the design scales well.

In [ ]:
#Part 3 - Data Modelling
# Generate simulation data 生成模拟数据
user_ids = [f"user{i:03d}" for i in range(1, 21)]
workout_types = ["running", "cycling", "strength", "yoga", "swimming"]
items = []
for uid in user_ids[:20]:  # 20 users
    num_workouts = random.randint(3, 10)
    workouts = []
    for _ in range(num_workouts):
        w_date = datetime.date.today() - datetime.timedelta(days=random.randint(1, 60))
        w_type = random.choice(workout_types)
        duration = random.randint(15, 90)
        calories = int(duration * (random.uniform(8, 12)))
        workouts.append({
            "date": w_date.isoformat(),
            "type": w_type,
            "duration": duration,
            "calories": calories
        })
    # Sort by date 按日期排序
    workouts.sort(key=lambda x: x["date"], reverse=True)
    items.append({
        "id": uid,
        "userId": uid,
        "name": f"User {uid[-3:]}",
        "age": random.randint(18, 65),
        "weight": round(random.uniform(50, 100), 1),
        "workouts": workouts
    })


In [ ]:
#Part 4 - Queries
# Insert data 插入数据
for item in items:
    container.upsert_item(item)
    print(f"Inserted {item['id']}")

print(f"Total items inserted: {len(items)}")

# Query 1: All Users (SELECT) 查询 1: 所有用户
all_users = list(container.query_items(query="SELECT * FROM c", enable_cross_partition_query=True))
print(f"\n1. Total users: {len(all_users)}")

# Query 2: Filter (WHERE) 查询 2: 过滤
user_query = "SELECT * FROM c WHERE c.userId = 'user001'"
user = list(container.query_items(query=user_query, enable_cross_partition_query=True))
print(f"\n2. User001: {user[0]['name']}")

# Query 3: Sort (ORDER BY) 查询 3: 排序
sorted_users = list(container.query_items(query="SELECT * FROM c ORDER BY c.age DESC", enable_cross_partition_query=True))
print(f"\n3. Oldest user age: {sorted_users[0]['age']}")

# Query 4: Aggregate COUNT 查询 4: 聚合 COUNT
count_query = "SELECT VALUE COUNT(1) FROM c"
count = list(container.query_items(query=count_query, enable_cross_partition_query=True))[0]
print(f"\n4. Total documents: {count}")

# Query 5: Nested Arrays (JOIN) 查询 5: 嵌套数组 (JOIN)
nested_query = "SELECT c.userId, w.type FROM c JOIN w IN c.workouts WHERE w.type = 'running'"
runners = list(container.query_items(query=nested_query, enable_cross_partition_query=True))
print(f"\n5. Users who did running: {len(runners)}")


Inserted user001
Inserted user002
Inserted user003
Inserted user004
Inserted user005
Inserted user006
Inserted user007
Inserted user008
Inserted user009
Inserted user010
Inserted user011
Inserted user012
Inserted user013
Inserted user014
Inserted user015
Inserted user016
Inserted user017
Inserted user018
Inserted user019
Inserted user020
Total items inserted: 20

1. Total users: 20

2. User001: User 001

3. Oldest user age: 63

4. Total documents: 20

5. Users who did running: 22


In [ ]:
#Part 5 - Python Intergration

# Example of a write operation: Add a new workout to user001 写操作示例: 添加新 workout 到 user001
user001_item = container.read_item(item="user001", partition_key="user001")
new_workout = {
    "date": datetime.date.today().isoformat(),
    "type": "walking",
    "duration": 20,
    "calories": 100
}
user001_item["workouts"].append(new_workout)
container.upsert_item(user001_item)
print(f"\nWrite operation: Added walking workout to user001")


Write operation: Added walking workout to user001


#Part 6 - Reflection


**1. Why did you structure your data the way you did?**  
I structured the data by embedding each user’s workouts as an array inside the user document. This matches the main access pattern: retrieving a user’s full profile together with their workout history in a single read operation. It avoids separate queries or client‑side joins.

**2. Why did you choose your partition key (`/userId`)?**  
I chose `/userId` as the partition key because most queries are user‑centric (e.g., “get user X and all their workouts”). This ensures all data for a single user stays in the same logical partition, making reads fast and cost‑efficient. It also distributes data evenly across partitions as long as the number of users is large.

**3. What challenges did you encounter?**  
The main challenge was generating realistic nested data (workouts) and ensuring the queries on nested arrays (e.g., `JOIN` to find users who did “running”) worked correctly. I also had to decide how many workouts to embed per user – too many could exceed the document size limit, but for a fitness app with typical usage (dozens of workouts) it is fine.

**4. What would happen if your system scaled?**  
If the system scaled to millions of users, the partition key `/userId` would still distribute data evenly, because each user is a separate partition key value. However, if a single user recorded an extremely large number of workouts (e.g., thousands), that one partition could become a “hot partition”. For normal usage, the design scales well. Queries that scan all users (like `SELECT * FROM c`) would become slower and more expensive, so they should be avoided in production.

**5. Would you redesign anything?**  
Yes, I would make a few improvements:
- Use a more realistic data model: store `workouts` as a separate container if users record hundreds of workouts, but for the current lab the embedded design is fine.
- Add a composite index to speed up queries on workout `type`.
- Implement pagination for large result sets instead of loading all users at once.
- Use a TTL (time‑to‑live) for old workout data if not needed forever.